## Required Imports and Constants

In [20]:
# IMPORTS
from dataset_manager import AGAIN_Manager, RECOLA_Manager
from keras.models import Sequential
from keras.layers import LSTM, Dense, Input
from keras.optimizers import Adam
import numpy as np
import os
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix, mean_squared_error
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import initializers
import time
from typing import Dict, List
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

In [21]:
# CONSTANTS
AGAIN_REPORT_FILE = "Results/AGAIN_Results.xlsx"
NUMBER_OF_DECIMALS = 5
RECOLA_REPORT_FILE = "Results/RECOLA_Results.xlsx"
TEMP_INPUT = "temporary_dataset.csv"
TEMP_OUTPUT = "temporary_invariant_features.csv"

SEED = 42

## Model Training Blocks

In [22]:
manager_AGAIN = AGAIN_Manager(reload_datasets=False)
manager_RECOLA = RECOLA_Manager(reload_dataset=False)

In [23]:
def confusion_matrix_calculations(conf_matrix: np.ndarray):
    true_neg, false_pos, false_neg, true_pos = conf_matrix.ravel()
    
    accuracy = round(float((true_pos + true_neg) / np.sum(conf_matrix)), NUMBER_OF_DECIMALS)
    precision = round(float(true_pos / (true_pos + false_pos) if (true_pos + false_pos) else 0), NUMBER_OF_DECIMALS)
    recall = round(float(true_pos / (true_pos + false_neg) if (true_pos + false_neg) else 0), NUMBER_OF_DECIMALS)

    return accuracy, precision, recall

def save_results(report_file: str, test_name: str, details_to_save: List):
    # STEP 1: DECLARING WORKBOOK COLUMNS
    report_columns = ["Test Name", 
                      "Standard Average Feature Count", "Invariant Average Feature Count", "PCA Average Feature Count",
                      "Standard Accuracy", "Invariant Accuracy", "PCA Accuracy",
                      "Standard Precision", "Invariant Precision", "PCA Precision",
                      "Standard Recall", "Invariant Recall", "PCA Recall",
                      "Standard PCC", "Invariant PCC", "PCA PCC"]
    per_fold_columns = ["Test Name", "Fold Number", "Accuracy", "Precision", "Recall"]

    # If file doesn't exist, create it with all sheets and headers
    if not os.path.exists(report_file):
        report_lr = pd.DataFrame(columns=report_columns)
        report_nn = pd.DataFrame(columns=report_columns)
        report_lstm = pd.DataFrame(columns=report_columns)
        report_pflr = pd.DataFrame(columns=per_fold_columns)
        report_pfnn = pd.DataFrame(columns=per_fold_columns)
        report_pflstm = pd.DataFrame(columns=per_fold_columns)
        with pd.ExcelWriter(report_file, engine='xlsxwriter') as writer:
            report_lr.to_excel(writer, sheet_name='Experiment Reports (LR)', index=False)
            report_nn.to_excel(writer, sheet_name='Experiment Reports (NN)', index=False)
            report_lstm.to_excel(writer, sheet_name='Experiment Reports (LSTM)', index=False)
            report_pflr.to_excel(writer, sheet_name='Per Fold Results (LR)', index=False)
            report_pfnn.to_excel(writer, sheet_name='Per Fold Results (NN)', index=False)
            report_pflstm.to_excel(writer, sheet_name='Per Fold Results (LSTM)', index=False)

    # STEP 2: UNPACK RESULTS
    log_stan, log_inv, log_pca, nn_stan, nn_inv, nn_pca, lstm_stan, lstm_inv, lstm_pca = details_to_save

    def update_sheet(sheet_name, entry, columns, primary_key):
        if isinstance(entry, dict):
            entry = [entry]
        try:
            existing = pd.read_excel(report_file, sheet_name=sheet_name)
        except Exception:
            existing = pd.DataFrame(columns=columns)

        new_rows = pd.DataFrame([{col: e.get(col, None) for col in columns} for e in entry])

        if not existing.empty:
            existing = existing.merge(
                new_rows[primary_key],
                on=primary_key,
                how="left",
                indicator=True
            )
            existing = existing[existing["_merge"] == "left_only"].drop(columns="_merge")

        updated = pd.concat([existing, new_rows], ignore_index=True)

        with pd.ExcelWriter(report_file, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
            updated.to_excel(writer, sheet_name=sheet_name, index=False)

    # STEP 3: SAVE LOGISTIC REGRESSION RESULTS
    entry_lr = {
        "Test Name": test_name,
        "Standard Average Feature Count": log_stan["feature_count"],
        "Invariant Average Feature Count": log_inv["feature_count"],
        "PCA Average Feature Count": log_pca["feature_count"],
        "Standard Accuracy": log_stan["accuracy"],
        "Invariant Accuracy": log_inv["accuracy"],
        "PCA Accuracy": log_pca["accuracy"],
        "Standard Precision": log_stan["precision"],
        "Invariant Precision": log_inv["precision"],
        "PCA Precision": log_pca["precision"],
        "Standard Recall": log_stan["recall"],
        "Invariant Recall": log_inv["recall"],
        "PCA Recall": log_pca["recall"],
        "Standard PCC": log_stan["pcc"],
        "Invariant PCC": log_inv["pcc"],
        "PCA PCC": log_pca["pcc"],
    }
    update_sheet("Experiment Reports (LR)", entry_lr, report_columns, ["Test Name"])

    # STEP 4: SAVE NEURAL NETWORK RESULTS
    entry_nn = {
        "Test Name": test_name,
        "Standard Average Feature Count": nn_stan["feature_count"],
        "Invariant Average Feature Count": nn_inv["feature_count"],
        "PCA Average Feature Count": nn_pca["feature_count"],
        "Standard Accuracy": nn_stan["accuracy"],
        "Invariant Accuracy": nn_inv["accuracy"],
        "PCA Accuracy": nn_pca["accuracy"],
        "Standard Precision": nn_stan["precision"],
        "Invariant Precision": nn_inv["precision"],
        "PCA Precision": nn_pca["precision"],
        "Standard Recall": nn_stan["recall"],
        "Invariant Recall": nn_inv["recall"],
        "PCA Recall": nn_pca["recall"],
        "Standard PCC": nn_stan["pcc"],
        "Invariant PCC": nn_inv["pcc"],
        "PCA PCC": nn_pca["pcc"],
    }
    update_sheet("Experiment Reports (NN)", entry_nn, report_columns, ["Test Name"])

    # STEP 5: SAVE LSTM RESULTS
    entry_lstm = {
        "Test Name": test_name,
        "Standard Average Feature Count": lstm_stan["feature_count"],
        "Invariant Average Feature Count": lstm_inv["feature_count"],
        "PCA Average Feature Count": lstm_pca["feature_count"],
        "Standard Accuracy": lstm_stan["accuracy"],
        "Invariant Accuracy": lstm_inv["accuracy"],
        "PCA Accuracy": lstm_pca["accuracy"],
        "Standard Precision": lstm_stan["precision"],
        "Invariant Precision": lstm_inv["precision"],
        "PCA Precision": lstm_pca["precision"],
        "Standard Recall": lstm_stan["recall"],
        "Invariant Recall": lstm_inv["recall"],
        "PCA Recall": lstm_pca["recall"],
        "Standard PCC": lstm_stan["pcc"],
        "Invariant PCC": lstm_inv["pcc"],
        "PCA PCC": lstm_pca["pcc"],
    }
    update_sheet("Experiment Reports (LSTM)", entry_lstm, report_columns, ["Test Name"])

    # STEP 6: PER FOLD RESULTS FOR ALL MODELS
    def log_folds(sheet_name, tag_prefix, test_data):
        for variant, tag in zip(test_data, ["standard", "invariant", "pca"]):
            for fold in variant["fold results"]:
                fold_entry = {
                    "Test Name": f"{test_name}_{tag}",
                    "Fold Number": fold,
                    "Accuracy": variant["fold results"][fold]["accuracy"],
                    "Precision": variant["fold results"][fold]["precision"],
                    "Recall": variant["fold results"][fold]["recall"],
                }
                update_sheet(sheet_name, fold_entry, per_fold_columns, ["Test Name", "Fold Number"])

    log_folds("Per Fold Results (LR)", test_name, [log_stan, log_inv, log_pca])
    log_folds("Per Fold Results (NN)", test_name, [nn_stan, nn_inv, nn_pca])
    log_folds("Per Fold Results (LSTM)", test_name, [lstm_stan, lstm_inv, lstm_pca])


In [24]:
def logistic_regresion_test(dataset_name: str, data_pack: Dict, feature_count: str, invariant_flag: bool = False, pca_flag: bool = False):
    # STEP 0: DECLARING FOLD LISTS   
    conf_matrix_list = []
    pcc_list = []
    feature_count_list = []

    # STEP 1: UNPACK DATA FOR MODEL
    dataset = data_pack["dataset"]
    groups = data_pack["groups"]
    folds = data_pack["folds"]
    features = data_pack["features"]
    target = data_pack["target"]
    continuous_target = data_pack["continuous_target"]

    # STEP 2: RUNNING THE MODEL WITH CROSS VALIDATION SCHEME
    group_k_fold = GroupKFold(n_splits=folds)
    fold_counter = 1
    for train_index, test_index in group_k_fold.split(features, target, groups):

        # STEP 2.1: CHECK FOR ALTERNATE TESTS
        if invariant_flag is True:
            fold_dataframe = dataset.iloc[train_index]
            fold_dataframe.to_csv("temporary_dataset.csv")
            
            if dataset_name == "RECOLA":
                invariant_features = manager_RECOLA.get_invariant_features(input_file=TEMP_INPUT, output_file=TEMP_OUTPUT, invariant_feature_count=feature_count)
            elif dataset_name == "AGAIN":
                invariant_features = manager_AGAIN.get_invariant_features(input_file=TEMP_INPUT, output_file=TEMP_OUTPUT, invariant_feature_count=feature_count)
            fold_features = features[invariant_features]

        elif pca_flag is True:
            scaler = StandardScaler()
            fold_features_scaled = scaler.fit_transform(features)

            pca = PCA(n_components=int(feature_count))
            fold_features_pca = pca.fit_transform(fold_features_scaled)

            fold_features = pd.DataFrame(fold_features_pca, index=features.index)

        elif invariant_flag is False and pca_flag is False:
            fold_features = features

        feature_count_list.append(fold_features.shape[1])

        # STEP 2.2: IF THERE ARE STILL FEATURES, GO THROUGH THE TRAINING PROCESS
        if not fold_features.empty:
            # STEP 2.3: OBTAINING TRAINING DATA
            feature_train = fold_features.iloc[train_index]
            target_train = target.iloc[train_index]

            # STEP 2.4: OBTAINING TESTING DATA FOR FOLD
            feature_test = fold_features.iloc[test_index]
            target_test = target.iloc[test_index]
            contin_target_test = continuous_target.iloc[test_index]

            # STEP 2.5: TRAINING THE MODEL
            model = LogisticRegression(solver='liblinear', max_iter=1000)
            model.fit(X=feature_train, y=target_train)

            # STEP 2.6: PREDICTIONS ON THE MODEL
            test_prediction = model.predict(feature_test)
            test_porb_prediction = model.predict_proba(feature_test)[:, 1]

            # STEP 2.7: GETTING EVALUATION METRICS
            pcc, _ = stats.pearsonr(test_porb_prediction, contin_target_test)
            pcc_list.append(pcc)

            conf_matrix = confusion_matrix(target_test, test_prediction)
            conf_matrix_list.append(conf_matrix) 

        fold_counter += 1

    # STEP 3: CALCULATING METRIC RESULTS FOR TEST
    results = {}
    
    # STEP 3.1: GET AVERAGE CONFUSION MATRIX RESULTS
    if conf_matrix_list:
        mean_conf_matrix = np.mean(conf_matrix_list, axis=0)
        results["accuracy"], results["precision"], results["recall"] = confusion_matrix_calculations(conf_matrix=mean_conf_matrix)
    else:
        results["accuracy"], results["precision"], results["recall"] = "NaN", "NaN", "NaN" 

    # STEP 3.2: GET AVERAGE PCC RESULT
    if pcc_list:
        results["pcc"] = round(float(np.mean(pcc_list)), NUMBER_OF_DECIMALS)
    else:
        results["pcc"] = "NaN"
        
    # STEP 3.3: GET PER FOLD RESULTS
    fold_results = {}
    fold_count = 1
    for matrix in conf_matrix_list:
        fold_metrics = {}
        fold_metrics["accuracy"], fold_metrics["precision"], fold_metrics["recall"] = confusion_matrix_calculations(conf_matrix=matrix)
        fold_results[f"Fold {fold_count}"] = fold_metrics
        fold_count += 1
    results["fold results"] = fold_results

    # STEP 3.4: GET FEATURE COUNT
    results["feature_count"] = sum(feature_count_list) / len(feature_count_list)
    
    return results

def run_logistic_regression_tests(dataset_name: str, model_data: Dict, feature_count: str):
    # STEP 1: RUN STANDARD TEST
    print("- Logistic Regression", end=""); start_time = time.time()
    standard_results = logistic_regresion_test(dataset_name=dataset_name, data_pack=model_data, feature_count=feature_count)

    # STEP 2: RUN INVARIANT FEATURES TEST
    invariant_results = logistic_regresion_test(dataset_name=dataset_name, data_pack=model_data, feature_count=feature_count, invariant_flag=True)
        
    # STEP 3: RUN PCA TEST
    pca_results = logistic_regresion_test(dataset_name=dataset_name, data_pack=model_data, feature_count=feature_count, pca_flag=True)
    print(f"- Execution Time: {(time.time() - start_time):.2f} seconds")

    return standard_results, invariant_results, pca_results

In [25]:
def neural_network_test(dataset_name: str, data_pack: Dict, feature_count: str, invariant_flag: bool = False, pca_flag: bool = False):
    # STEP 0: DECLARING FOLD LISTS   
    conf_matrix_list = []
    pcc_list = []
    feature_count_list = []

    # STEP 1: UNPACK DATA FOR MODEL
    dataset = data_pack["dataset"]
    groups = data_pack["groups"]
    folds = data_pack["folds"]
    features = data_pack["features"]
    target = data_pack["target"]
    continuous_target = data_pack["continuous_target"]

    # STEP 2: RUNNING THE MODEL WITH CROSS VALIDATION SCHEME
    group_k_fold = GroupKFold(n_splits=folds)
    fold_counter = 1
    for train_index, test_index in group_k_fold.split(features, target, groups):
        # STEP 2.1: IF AN INVARIANT TEST, OBTAIN INVARIANT FEATURES
        if invariant_flag is True:
            fold_dataframe = dataset.iloc[train_index]
            fold_dataframe.to_csv("temporary_dataset.csv")
            
            if dataset_name == "RECOLA":
                invariant_features = manager_RECOLA.get_invariant_features(input_file=TEMP_INPUT, output_file=TEMP_OUTPUT, invariant_feature_count=feature_count)
            elif dataset_name == "AGAIN":
                invariant_features = manager_AGAIN.get_invariant_features(input_file=TEMP_INPUT, output_file=TEMP_OUTPUT, invariant_feature_count=feature_count)
            fold_features = features[invariant_features]

        elif pca_flag is True:
            scaler = StandardScaler()
            fold_features_scaled = scaler.fit_transform(features)

            pca = PCA(n_components=int(feature_count))
            fold_features_pca = pca.fit_transform(fold_features_scaled)

            fold_features = pd.DataFrame(fold_features_pca, index=features.index)

        elif invariant_flag is False and pca_flag is False:
            fold_features = features

        feature_count_list.append(fold_features.shape[1])

        # STEP 2.2: IF THERE ARE STILL FEATURES, GO THROUGH THE TRAINING PROCESS
        if not fold_features.empty:
            # STEP 2.3: OBTAINING TRAINING DATA
            feature_train = fold_features.iloc[train_index]
            target_train = target.iloc[train_index]

            # STEP 2.4: OBTAINING TESTING DATA FOR FOLD
            feature_test = fold_features.iloc[test_index]
            target_test = target.iloc[test_index]
            contin_target_test = continuous_target.iloc[test_index]

            # STEP 2.5: TRAINING THE MODEL
            model = MLPClassifier(hidden_layer_sizes=(32,), max_iter=10000, random_state=42)
            model.fit(X=feature_train, y=target_train)

            # STEP 2.6: PREDICTIONS ON THE MODEL
            test_prediction = model.predict(feature_test)
            test_porb_prediction = model.predict_proba(feature_test)[:, 1]

            # STEP 2.7: GETTING EVALUATION METRICS
            pcc, _ = stats.pearsonr(test_porb_prediction, contin_target_test)
            pcc_list.append(pcc)

            conf_matrix = confusion_matrix(target_test, test_prediction)
            conf_matrix_list.append(conf_matrix) 

        fold_counter += 1

    # STEP 3: CALCULATING METRIC RESULTS FOR TEST
    results = {}
    
    # STEP 3.1: GET AVERAGE CONFUSION MATRIX RESULTS
    if conf_matrix_list:
        mean_conf_matrix = np.mean(conf_matrix_list, axis=0)
        results["accuracy"], results["precision"], results["recall"] = confusion_matrix_calculations(conf_matrix=mean_conf_matrix)
    else:
        results["accuracy"], results["precision"], results["recall"] = "NaN", "NaN", "NaN" 

    # STEP 3.2: GET AVERAGE PCC RESULT
    if pcc_list:
        results["pcc"] = round(float(np.mean(pcc_list)), NUMBER_OF_DECIMALS)
    else:
        results["pcc"] = "NaN"

    # STEP 3.3: GET PER FOLD RESULTS
    fold_results = {}
    fold_count = 1
    for matrix in conf_matrix_list:
        fold_metrics = {}
        fold_metrics["accuracy"], fold_metrics["precision"], fold_metrics["recall"] = confusion_matrix_calculations(conf_matrix=matrix)
        fold_results[f"Fold {fold_count}"] = fold_metrics
        fold_count += 1
    results["fold results"] = fold_results

    # STEP 3.4: GET FEATURE COUNT
    results["feature_count"] = sum(feature_count_list) / len(feature_count_list)
    
    return results

def run_neural_network_tests(dataset_name: str, model_data: Dict, feature_count: str):
    # STEP 1: RUN STANDARD TEST
    print("- Neural Networks", end=""); start_time = time.time()
    standard_results = neural_network_test(dataset_name=dataset_name, data_pack=model_data, feature_count=feature_count)

    # STEP 2: RUN INVARIANT FEATURES TEST
    invariant_results = neural_network_test(dataset_name=dataset_name, data_pack=model_data, feature_count=feature_count, invariant_flag=True)
    
    # # STEP 3: RUN PCA TEST
    pca_results = neural_network_test(dataset_name=dataset_name, data_pack=model_data, feature_count=feature_count, pca_flag=True)
    print(f"- Execution Time: {(time.time() - start_time):.2f} seconds")

    return standard_results, invariant_results, pca_results

In [26]:
def lstm_test(dataset_name: str, data_pack: Dict, feature_count: str, invariant_flag: bool = False, pca_flag: bool = False, timesteps: int = 10, threshold: float = 0.5):
    # STEP 0: DECLARING METRIC LISTS
    conf_matrix_list = []
    pcc_list = []
    feature_count_list = []

    # STEP 1: UNPACK DATA
    dataset = data_pack["dataset"]
    groups = data_pack["groups"]
    folds = data_pack["folds"]
    features = data_pack["features"]
    target = data_pack["target"]
    continuous_target = data_pack["continuous_target"]

    def LSTM__transform(X, y, timesteps):
        num_samples = X.shape[0] - timesteps
        if num_samples <= 0:
            return np.array([]), np.array([])
        X_seq = np.array([X[i:i+timesteps] for i in range(num_samples)])
        y_seq = np.array([y[i+timesteps] for i in range(num_samples)])
        return X_seq, y_seq

    group_k_fold = GroupKFold(n_splits=folds)
    fold_counter = 1

    for train_index, test_index in group_k_fold.split(features, target, groups):

        # STEP 2.1: HANDLE FEATURE TRANSFORMATION
        if invariant_flag:
            fold_dataframe = dataset.iloc[train_index]
            fold_dataframe.to_csv("temporary_dataset.csv")

            if dataset_name == "RECOLA":
                invariant_features = manager_RECOLA.get_invariant_features(
                    input_file=TEMP_INPUT, output_file=TEMP_OUTPUT, invariant_feature_count=feature_count)
            elif dataset_name == "AGAIN":
                invariant_features = manager_AGAIN.get_invariant_features(
                    input_file=TEMP_INPUT, output_file=TEMP_OUTPUT, invariant_feature_count=feature_count)
            fold_features = features[invariant_features]

        elif pca_flag:
            scaler = StandardScaler()
            fold_features_scaled = scaler.fit_transform(features)
            pca = PCA(n_components=int(feature_count))
            fold_features_pca = pca.fit_transform(fold_features_scaled)
            fold_features = pd.DataFrame(fold_features_pca, index=features.index)

        else:
            fold_features = features

        feature_count_list.append(fold_features.shape[1])

        if not fold_features.empty:
            # STEP 2.3: OBTAIN TRAIN & TEST SETS
            X = fold_features.values
            y = continuous_target.values

            X_train = X[train_index]
            y_train = y[train_index]
            X_test = X[test_index]
            y_test = y[test_index]
            y_bin_test = target.iloc[test_index].values  # For confusion matrix

            # STEP 2.4: RESHAPE FOR LSTM
            X_train_seq, y_train_seq = LSTM__transform(X_train, y_train, timesteps)
            X_test_seq, y_test_seq = LSTM__transform(X_test, y_test, timesteps)
            y_bin_test_seq = y_bin_test[timesteps:]  # Align binary target with test_seq

            if X_train_seq.size == 0 or X_test_seq.size == 0:
                continue

            # STEP 2.5: BUILD & TRAIN LSTM
            model = Sequential()
            model.add(Input(shape=(timesteps, X_train_seq.shape[2])))
            model.add(LSTM(
                units=12,
                return_sequences=False,
                kernel_initializer=initializers.GlorotUniform(seed=SEED),
                recurrent_initializer=initializers.Orthogonal(seed=SEED),
                bias_initializer=initializers.Zeros()
            ))
            model.add(Dense(
                1,
                kernel_initializer=initializers.GlorotUniform(seed=SEED),
                bias_initializer=initializers.Zeros()
            ))

            model.compile(optimizer=Adam(), loss='mean_squared_error', metrics=['mse'])
            model.fit(X_train_seq, y_train_seq, epochs=10, batch_size=32, verbose=0, shuffle=False)

            # STEP 2.6: PREDICT & EVALUATE
            y_pred = model.predict(X_test_seq, verbose=0).flatten()

            # PCC
            if len(y_pred) > 1:
                pcc, _ = stats.pearsonr(y_pred, y_test_seq)
                pcc_list.append(pcc)

            # Confusion matrix
            y_pred_class = (y_pred >= threshold).astype(int)
            y_true_class = (y_test_seq >= threshold).astype(int)

            conf_matrix = confusion_matrix(y_true_class, y_pred_class, labels=[0, 1])
            conf_matrix_list.append(conf_matrix)

        fold_counter += 1

    # STEP 3: AGGREGATE RESULTS
    results = {}

    if conf_matrix_list:
        mean_conf_matrix = np.mean(conf_matrix_list, axis=0)
        results["accuracy"], results["precision"], results["recall"] = confusion_matrix_calculations(mean_conf_matrix)
    else:
        results["accuracy"], results["precision"], results["recall"] = "NaN", "NaN", "NaN"

    if pcc_list:
        results["pcc"] = round(float(np.mean(pcc_list)), NUMBER_OF_DECIMALS)
    else:
        results["pcc"] = "NaN"

    # STEP 3.3: PER-FOLD METRICS
    fold_results = {}
    fold_count = 1
    for matrix in conf_matrix_list:
        fold_metrics = {}
        fold_metrics["accuracy"], fold_metrics["precision"], fold_metrics["recall"] = confusion_matrix_calculations(matrix)
        fold_results[f"Fold {fold_count}"] = fold_metrics
        fold_count += 1
    results["fold results"] = fold_results

    # STEP 3.4: AVERAGE FEATURE COUNT
    results["feature_count"] = sum(feature_count_list) / len(feature_count_list) if feature_count_list else 0

    return results

def run_lstm_tests(dataset_name: str, model_data: Dict, feature_count: str):
    # STEP 1: RUN STANDARD TEST
    print("- LSTM", end=""); start_time = time.time()
    standard_results = lstm_test(dataset_name=dataset_name, data_pack=model_data, feature_count=feature_count)

    # STEP 2: RUN INVARIANT FEATURES TEST
    invariant_results = lstm_test(dataset_name=dataset_name, data_pack=model_data, feature_count=feature_count, invariant_flag=True)
        
    # STEP 3: RUN PCA TEST
    pca_results = lstm_test(dataset_name=dataset_name, data_pack=model_data, feature_count=feature_count, pca_flag=True)
    print(f"- Execution Time: {(time.time() - start_time):.2f} seconds")

    return standard_results, invariant_results, pca_results

In [27]:
def checkpoint_checker(test_name: str) -> bool:
    checkpoint_file = "Results/[checkpoints].csv"
    if os.path.exists(checkpoint_file) is False:
        return True
    else:
        file = pd.read_csv(checkpoint_file)
        test_list = list(file["Test Name"])
        if test_name in test_list:
            return False
        else:
            return True

def checkpoint_updater(test_name: str):
    checkpoint_file = "Results/[checkpoints].csv"
    if os.path.exists(checkpoint_file) is False:
        with open(checkpoint_file, 'w') as f:
            pass

        file = pd.DataFrame({"Test Name": [test_name]})
        file.to_csv(path_or_buf=checkpoint_file, index=False)
    else:
        file = pd.read_csv(checkpoint_file)
        test_list = list(file["Test Name"])

        test_list.append(test_name)
        file = pd.DataFrame({"Test Name": test_list})
        file.to_csv(path_or_buf=checkpoint_file, index=False)

## AGAIN Experiments

In [28]:
DATASET_NAME = "AGAIN"

In [29]:
# TEST CASE 1: SINGLE GENRE, GAMES PER ENVIRONMENT
# STEP 1: DECLARE ITERATION PARAMETERS
max_num_invariant_features = ["6", "8", "10", "12", "14", "16"]
genre_datasets = ["again_platformer", "again_racing", "again_shooter"]

# STEP 2: SET UP THE TESTING LOOP
for genre in genre_datasets:
    for inv_feature_count in max_num_invariant_features:
        # STEP 2.1 CREATING TEST NAME
        case_name = f"AGAIN_Test_1_{genre[6:]}"
        test_name = f"{case_name}_{inv_feature_count}"
        
        # STEP 2.2: CHECKPOINT CHECK
        if checkpoint_checker(test_name=test_name) is True:
            print(f"Test Name: {test_name}")
            # STEP 2.3: LOAD/CREATE DATASET
            dataset = manager_AGAIN.load_dataset(dataset_name=genre.upper())
            dataset = dataset.dropna(axis=1)

            # STEP 2.4: DATA PREP FOR MODEL TRAINING
            data_for_model = {}
            data_for_model["dataset"] = dataset
            data_for_model["groups"] = list(dataset["[control]game"])
            data_for_model["folds"] = len(set(data_for_model["groups"]))
            data_for_model["features"]  = dataset.filter([x for x in list(dataset.columns) if x not in ["[control]player_id", "[control]genre", "[control]game", "[output]arousal", "Binary_Arousal_Class"]])    
            data_for_model["target"] = dataset["Binary_Arousal_Class"]
            data_for_model["continuous_target"] = dataset["[output]arousal"]

            if data_for_model["features"].shape[1] >= int(inv_feature_count):
                # STEP 2.5: RUN LOGISTIC REGRESSION TESTS
                log_stan_results, log_inv_results, log_pca_results = run_logistic_regression_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.6: RUN NEURAL NETWORK TESTS
                nn_stan_results, nn_inv_results, nn_pca_results = run_neural_network_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)
                
                # STEP 2.7: RUN LSTM TESTS
                lstm_stan, lstm_inv, lstm_pca = run_lstm_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.8: SAVING RESULTS AND UPDATING CHECKPOINT FILE
                save_results(report_file=AGAIN_REPORT_FILE, test_name=test_name, details_to_save=[log_stan_results, log_inv_results, log_pca_results, nn_stan_results, nn_inv_results, nn_pca_results, lstm_stan, lstm_inv, lstm_pca])
                checkpoint_updater(test_name=test_name)
            else:
                print(f"Less features than max_invariant_features ({data_for_model["features"].shape[1]})")
            print()

In [30]:
# TEST CASE 2: SINGLE GAME, PARTICIPANT GROUPS OF 5, 6 GROUPS
# STEP 1: DECLARE ITERATION PARAMETERS
max_num_invariant_features = ["6", "8", "10", "12", "14", "16"]
game_datasets = ["again_platformer_endless", "again_platformer_pirates!", "again_platformer_run'n'gun",
                    "again_racing_apexspeed", "again_racing_solid", "again_racing_tinycars",
                    "again_shooter_heist!", "again_shooter_shootout", "again_shooter_topdown",]

# STEP 2: SET UP THE TESTING LOOP
for game in game_datasets:
    for inv_feature_count in max_num_invariant_features:
        # STEP 2.1 CREATING TEST NAME
        case_name = f"AGAIN_Test_2_{game[6:]}"
        test_name = f"{case_name}_{inv_feature_count}"
        
        # STEP 2.2: CHECKPOINT CHECK
        if checkpoint_checker(test_name=test_name) is True:
            print(f"Test Name: {test_name}")
            # STEP 2.3: LOAD/CREATE DATASET
            dataset = manager_AGAIN.load_dataset(dataset_name=game.upper())
            dataset = manager_AGAIN.get_participant_groups(data=dataset, group_size=5, group_count=6)
            manager_AGAIN.save_test_dataset(dataset_name=f"{case_name}.csv", data=dataset)
            dataset = dataset.dropna(axis=1)

            # STEP 2.4: DATA PREP FOR MODEL TRAINING
            data_for_model = {}
            data_for_model["dataset"] = dataset
            data_for_model["groups"] = list(dataset["group_id"])
            data_for_model["folds"] = len(set(data_for_model["groups"]))
            data_for_model["features"]  = dataset.filter([x for x in list(dataset.columns) if x not in ["[control]player_id", "[control]genre", "[control]game", "[output]arousal", "Binary_Arousal_Class"]])    
            data_for_model["target"] = dataset["Binary_Arousal_Class"]
            data_for_model["continuous_target"] = dataset["[output]arousal"]

            if data_for_model["features"].shape[1] >= int(inv_feature_count):
                # STEP 2.5: RUN LOGISTIC REGRESSION TESTS
                log_stan_results, log_inv_results, log_pca_results = run_logistic_regression_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.6: RUN NEURAL NETWORK TESTS
                nn_stan_results, nn_inv_results, nn_pca_results = run_neural_network_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)            

                # STEP 2.7: RUN LSTM TESTS
                lstm_stan, lstm_inv, lstm_pca = run_lstm_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.8: SAVING RESULTS AND UPDATING CHECKPOINT FILE
                save_results(report_file=AGAIN_REPORT_FILE, test_name=test_name, details_to_save=[log_stan_results, log_inv_results, log_pca_results, nn_stan_results, nn_inv_results, nn_pca_results, lstm_stan, lstm_inv, lstm_pca])
                checkpoint_updater(test_name=test_name)
            else:
                print(f"Less features than max_invariant_features ({data_for_model["features"].shape[1]})")
            print()

## RECOLA Experiments

In [31]:
DATASET_NAME = "RECOLA"

In [32]:
# TEST CASE 1: MALE PARTICIPANTS, TARGET LABEL: AROUSAL
# STEP 1: DECLARE ITERATION PARAMETERS
max_num_invariant_features = ["6", "8", "10", "12", "14", "16"]
modalities = "Audio", "Video", "Physiology"

for modality in modalities:
    for inv_feature_count in max_num_invariant_features:
        # STEP 2.1 CREATING TEST NAME
        case_name = f"RECOLA_Test_1_{modality}"
        test_name = f"{case_name}_{inv_feature_count}"
        
        # STEP 2.2: CHECKPOINT CHECK
        if checkpoint_checker(test_name=test_name) is True:
            print(f"Test Name: {test_name}")

            # STEP 2.3: LOAD/CREATE DATASET
            dataset = manager_RECOLA.load_dataset(dataset_name="RECOLA_Base")
            dataset = manager_RECOLA.split_by_gender(data=dataset, gender="male")
            dataset = manager_RECOLA.keep_modality(data=dataset, modality=modality)
            dataset = manager_RECOLA.remove_class_label(label_to_keep="arousal", data=dataset)
            manager_RECOLA.save_test_dataset(dataset_name=f"{case_name}.csv", data=dataset)

            # STEP 2.4: DATA PREP FOR MODEL TRAINING
            data_for_model = {}
            data_for_model["dataset"] = dataset
            data_for_model["groups"] = list(dataset["Participant_Number"])
            data_for_model["folds"] = len(set(data_for_model["groups"]))
            data_for_model["features"]  = dataset.filter(regex=f'^{"ComPar"}|{"audio_speech"}|{"VIDEO"}|{"Face_detection"}|{"ECG"}|{"EDA"}', axis=1) 
            if "Class_Label_Valence" in dataset.columns: 
                data_for_model["target"]  = dataset["Class_Label_Valence"]
                data_for_model["continuous_target"] = dataset["Annotator_Valence"]
            elif "Class_Label_Arousal" in dataset.columns: 
                data_for_model["target"]  = dataset["Class_Label_Arousal"]
                data_for_model["continuous_target"] = dataset["Annotator_Arousal"]

            if data_for_model["features"].shape[1] >= int(inv_feature_count):
                # STEP 2.5: RUN LOGISTIC REGRESSION TESTS
                log_stan_results, log_inv_results, log_pca_results = run_logistic_regression_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.6: RUN NEURAL NETWORK TESTS
                nn_stan_results, nn_inv_results, nn_pca_results = run_neural_network_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.7: RUN LSTM TESTS
                lstm_stan, lstm_inv, lstm_pca = run_lstm_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.8: SAVING RESULTS AND UPDATING CHECKPOINT FILE
                save_results(report_file=RECOLA_REPORT_FILE, test_name=test_name, details_to_save=[log_stan_results, log_inv_results, log_pca_results, nn_stan_results, nn_inv_results, nn_pca_results, lstm_stan, lstm_inv, lstm_pca])
                checkpoint_updater(test_name=test_name)
            else:
                print(f"Less features than max_invariant_features ({data_for_model["features"].shape[1]})")     
            print() 

In [33]:
# TEST CASE 2: MALE PARTICIPANTS, TARGET LABEL: VALENCE
# STEP 1: DECLARE ITERATION PARAMETERS
max_num_invariant_features = ["6", "8", "10", "12", "14", "16"]
modalities = "Audio", "Video", "Physiology"

for modality in modalities:
    for inv_feature_count in max_num_invariant_features:
        # STEP 2.1 CREATING TEST NAME
        case_name = f"RECOLA_Test_2_{modality}"
        test_name = f"{case_name}_{inv_feature_count}"
        
        # STEP 2.2: CHECKPOINT CHECK
        if checkpoint_checker(test_name=test_name) is True:
            print(f"Test Name: {test_name}")

            # STEP 2.3: LOAD/CREATE DATASET
            dataset = manager_RECOLA.load_dataset(dataset_name="RECOLA_Base")
            dataset = manager_RECOLA.split_by_gender(data=dataset, gender="male")
            dataset = manager_RECOLA.keep_modality(data=dataset, modality=modality)
            dataset = manager_RECOLA.remove_class_label(label_to_keep="valence", data=dataset)
            manager_RECOLA.save_test_dataset(dataset_name=f"{case_name}.csv", data=dataset)

            # STEP 2.4: DATA PREP FOR MODEL TRAINING
            data_for_model = {}
            data_for_model["dataset"] = dataset
            data_for_model["groups"] = list(dataset["Participant_Number"])
            data_for_model["folds"] = len(set(data_for_model["groups"]))        
            data_for_model["features"]  = dataset.filter(regex=f'^{"ComPar"}|{"audio_speech"}|{"VIDEO"}|{"Face_detection"}|{"ECG"}|{"EDA"}', axis=1) 
            if "Class_Label_Valence" in dataset.columns: 
                data_for_model["target"]  = dataset["Class_Label_Valence"]
                data_for_model["continuous_target"] = dataset["Annotator_Valence"]
            elif "Class_Label_Arousal" in dataset.columns: 
                data_for_model["target"]  = dataset["Class_Label_Arousal"]
                data_for_model["continuous_target"] = dataset["Annotator_Arousal"]

            if data_for_model["features"].shape[1] >= int(inv_feature_count):
                # STEP 2.5: RUN LOGISTIC REGRESSION TESTS
                log_stan_results, log_inv_results, log_pca_results = run_logistic_regression_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.6: RUN NEURAL NETWORK TESTS
                nn_stan_results, nn_inv_results, nn_pca_results = run_neural_network_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.7: RUN LSTM TESTS
                lstm_stan, lstm_inv, lstm_pca = run_lstm_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.8: SAVING RESULTS AND UPDATING CHECKPOINT FILE
                save_results(report_file=RECOLA_REPORT_FILE, test_name=test_name, details_to_save=[log_stan_results, log_inv_results, log_pca_results, nn_stan_results, nn_inv_results, nn_pca_results, lstm_stan, lstm_inv, lstm_pca])
                checkpoint_updater(test_name=test_name)
            else:
                print(f"Less features than max_invariant_features ({data_for_model["features"].shape[1]})")            
            print()

In [34]:
# TEST CASE 3: FEMALE PARTICIPANTS, TARGET LABEL: AROUSAL
# STEP 1: DECLARE ITERATION PARAMETERS
max_num_invariant_features = ["6", "8", "10", "12", "14", "16"]
modalities = "Audio", "Video", "Physiology"

for modality in modalities:
    for inv_feature_count in max_num_invariant_features:
        # STEP 2.1 CREATING TEST NAME
        case_name = f"RECOLA_Test_3_{modality}"
        test_name = f"{case_name}_{inv_feature_count}"
        
        # STEP 2.2: CHECKPOINT CHECK
        if checkpoint_checker(test_name=test_name) is True:
            print(f"Test Name: {test_name}")

            # STEP 2.3: LOAD/CREATE DATASET
            dataset = manager_RECOLA.load_dataset(dataset_name="RECOLA_Base")
            dataset = manager_RECOLA.split_by_gender(data=dataset, gender="female")
            dataset = manager_RECOLA.keep_modality(data=dataset, modality=modality)
            dataset = manager_RECOLA.remove_class_label(label_to_keep="arousal", data=dataset)
            manager_RECOLA.save_test_dataset(dataset_name=f"{case_name}.csv", data=dataset)

            # STEP 2.4: DATA PREP FOR MODEL TRAINING
            data_for_model = {}
            data_for_model["dataset"] = dataset
            data_for_model["groups"] = list(dataset["Participant_Number"])
            data_for_model["folds"] = len(set(data_for_model["groups"]))
            data_for_model["features"]  = dataset.filter(regex=f'^{"ComPar"}|{"audio_speech"}|{"VIDEO"}|{"Face_detection"}|{"ECG"}|{"EDA"}', axis=1) 
            if "Class_Label_Valence" in dataset.columns: 
                data_for_model["target"]  = dataset["Class_Label_Valence"]
                data_for_model["continuous_target"] = dataset["Annotator_Valence"]
            elif "Class_Label_Arousal" in dataset.columns: 
                data_for_model["target"]  = dataset["Class_Label_Arousal"]
                data_for_model["continuous_target"] = dataset["Annotator_Arousal"]

            if data_for_model["features"].shape[1] >= int(inv_feature_count):
                # STEP 2.5: RUN LOGISTIC REGRESSION TESTS
                log_stan_results, log_inv_results, log_pca_results = run_logistic_regression_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.6: RUN NEURAL NETWORK TESTS
                nn_stan_results, nn_inv_results, nn_pca_results = run_neural_network_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.7: RUN LSTM TESTS
                lstm_stan, lstm_inv, lstm_pca = run_lstm_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.8: SAVING RESULTS AND UPDATING CHECKPOINT FILE
                save_results(report_file=RECOLA_REPORT_FILE, test_name=test_name, details_to_save=[log_stan_results, log_inv_results, log_pca_results, nn_stan_results, nn_inv_results, nn_pca_results, lstm_stan, lstm_inv, lstm_pca])
                checkpoint_updater(test_name=test_name)
            else:
                print(f"Less features than max_invariant_features ({data_for_model["features"].shape[1]})")            
            print()

In [35]:
# TEST CASE 4: FEMALE PARTICIPANTS, TARGET LABEL: VALENCE
# STEP 1: DECLARE ITERATION PARAMETERS
max_num_invariant_features = ["6", "8", "10", "12", "14", "16"]
modalities = "Audio", "Video", "Physiology"

for modality in modalities:
    for inv_feature_count in max_num_invariant_features:
        # STEP 2.1 CREATING TEST NAME
        case_name = f"RECOLA_Test_4_{modality}"
        test_name = f"{case_name}_{inv_feature_count}"
        
        # STEP 2.2: CHECKPOINT CHECK
        if checkpoint_checker(test_name=test_name) is True:
            print(f"Test Name: {test_name}")

            # STEP 2.3: LOAD/CREATE DATASET
            dataset = manager_RECOLA.load_dataset(dataset_name="RECOLA_Base")
            dataset = manager_RECOLA.split_by_gender(data=dataset, gender="female")
            dataset = manager_RECOLA.keep_modality(data=dataset, modality=modality)
            dataset = manager_RECOLA.remove_class_label(label_to_keep="valence", data=dataset)
            manager_RECOLA.save_test_dataset(dataset_name=f"{case_name}.csv", data=dataset)

            # STEP 2.4: DATA PREP FOR MODEL TRAINING
            data_for_model = {}
            data_for_model["dataset"] = dataset
            data_for_model["groups"] = list(dataset["Participant_Number"])
            data_for_model["folds"] = len(set(data_for_model["groups"]))
            data_for_model["features"]  = dataset.filter(regex=f'^{"ComPar"}|{"audio_speech"}|{"VIDEO"}|{"Face_detection"}|{"ECG"}|{"EDA"}', axis=1) 
            if "Class_Label_Valence" in dataset.columns: 
                data_for_model["target"]  = dataset["Class_Label_Valence"]
                data_for_model["continuous_target"] = dataset["Annotator_Valence"]
            elif "Class_Label_Arousal" in dataset.columns: 
                data_for_model["target"]  = dataset["Class_Label_Arousal"]
                data_for_model["continuous_target"] = dataset["Annotator_Arousal"]

            if data_for_model["features"].shape[1] >= int(inv_feature_count):
                # STEP 2.5: RUN LOGISTIC REGRESSION TESTS
                log_stan_results, log_inv_results, log_pca_results = run_logistic_regression_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.6: RUN NEURAL NETWORK TESTS
                nn_stan_results, nn_inv_results, nn_pca_results = run_neural_network_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.7: RUN LSTM TESTS
                lstm_stan, lstm_inv, lstm_pca = run_lstm_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

                # STEP 2.8: SAVING RESULTS AND UPDATING CHECKPOINT FILE
                save_results(report_file=RECOLA_REPORT_FILE, test_name=test_name, details_to_save=[log_stan_results, log_inv_results, log_pca_results, nn_stan_results, nn_inv_results, nn_pca_results, lstm_stan, lstm_inv, lstm_pca])
                checkpoint_updater(test_name=test_name)
            else:
                print(f"Less features than max_invariant_features ({data_for_model["features"].shape[1]})")            
            print()

In [36]:
# # TEST CASE 5: DEPRECATED

# # TEST CASE 5: MALE AND FEMALE ENVIRONS (2), TARGET LABEL: AROUSAL
# # STEP 1: DECLARE ITERATION PARAMETERS
# max_num_invariant_features = ["6", "8", "10", "12", "14", "16"]

# for inv_feature_count in max_num_invariant_features:
#     # STEP 2.1 CREATING TEST NAME
#     case_name = "RECOLA_Test_5"
#     test_name = f"{case_name}_{inv_feature_count}"
    
#     # STEP 2.2: CHECKPOINT CHECK
#     if checkpoint_checker(test_name=test_name) is True:
#         print(f"Test Name: {test_name}")

#         # STEP 2.3: LOAD/CREATE DATASET
#         dataset = manager_RECOLA.load_dataset(dataset_name="RECOLA_Base")
#         male_dataset = manager_RECOLA.split_by_gender(data=dataset, gender="male")
#         female_dataset = manager_RECOLA.split_by_gender(data=dataset, gender="female")
#         dataset = manager_RECOLA.simplify_environs(datasets=[male_dataset, female_dataset])
#         dataset = manager_RECOLA.remove_class_label(label_to_keep="arousal", data=dataset)
#         manager_RECOLA.save_test_dataset(dataset_name=f"{case_name}.csv", data=dataset)

#         # STEP 2.4: DATA PREP FOR MODEL TRAINING
#         data_for_model = {}
#         data_for_model["dataset"] = dataset
#         data_for_model["groups"] = list(dataset["Participant_Number"])
#         data_for_model["folds"] = len(set(data_for_model["groups"]))
#         data_for_model["features"]  = dataset.filter(regex=f'^{"ComPar"}|{"audio_speech"}|{"VIDEO"}|{"Face_detection"}|{"ECG"}|{"EDA"}', axis=1) 
#         if "Class_Label_Valence" in dataset.columns: 
#             data_for_model["target"]  = dataset["Class_Label_Valence"]
#             data_for_model["continuous_target"] = dataset["Annotator_Valence"]
#         elif "Class_Label_Arousal" in dataset.columns: 
#             data_for_model["target"]  = dataset["Class_Label_Arousal"]
#             data_for_model["continuous_target"] = dataset["Annotator_Arousal"]

#         if data_for_model["features"].shape[1] >= int(inv_feature_count):
#             # STEP 2.5: RUN LOGISTIC REGRESSION TESTS
#             log_stan_results, log_inv_results, log_pca_results = run_logistic_regression_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

#             # STEP 2.6: RUN NEURAL NETWORK TESTS
#             nn_stan_results, nn_inv_results, nn_pca_results = run_neural_network_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

#             # STEP 2.7: RUN LSTM TESTS
#             lstm_stan, lstm_inv, lstm_pca = run_lstm_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

#             # STEP 2.8: SAVING RESULTS AND UPDATING CHECKPOINT FILE
#             save_results(report_file=AGAIN_REPORT_FILE, test_name=test_name, details_to_save=[log_stan_results, log_inv_results, log_pca_results, nn_stan_results, nn_inv_results, nn_pca_results, lstm_stan, lstm_inv, lstm_pca])
#             checkpoint_updater(test_name=test_name)
#         else:
#             print(f"Less features than max_invariant_features ({data_for_model["features"].shape[1]})")            
#         print()

In [37]:
# # TEST CASE 6: DEPRECATED

# #  TEST CASE 6: MALE AND FEMALE ENVIRONS (2), TARGET LABEL: VALENCE
# # STEP 1: DECLARE ITERATION PARAMETERS
# max_num_invariant_features = ["6", "8", "10", "12", "14", "16"]

# for inv_feature_count in max_num_invariant_features:
#     # STEP 2.1 CREATING TEST NAME
#     case_name = "RECOLA_Test_6"
#     test_name = f"{case_name}_{inv_feature_count}"
    
#     # STEP 2.2: CHECKPOINT CHECK
#     if checkpoint_checker(test_name=test_name) is True:
#         print(f"Test Name: {test_name}")

#         # STEP 2.3: LOAD/CREATE DATASET
#         dataset = manager_RECOLA.load_dataset(dataset_name="RECOLA_Base")
#         male_dataset = manager_RECOLA.split_by_gender(data=dataset, gender="male")
#         female_dataset = manager_RECOLA.split_by_gender(data=dataset, gender="female")
#         dataset = manager_RECOLA.simplify_environs(datasets=[male_dataset, female_dataset])
#         dataset = manager_RECOLA.remove_class_label(label_to_keep="valence", data=dataset)
#         manager_RECOLA.save_test_dataset(dataset_name=f"{case_name}.csv", data=dataset)

#         # STEP 2.4: DATA PREP FOR MODEL TRAINING
#         data_for_model = {}
#         data_for_model["dataset"] = dataset
#         data_for_model["groups"] = list(dataset["Participant_Number"])
#         data_for_model["folds"] = len(set(data_for_model["groups"]))
#         data_for_model["features"]  = dataset.filter(regex=f'^{"ComPar"}|{"audio_speech"}|{"VIDEO"}|{"Face_detection"}|{"ECG"}|{"EDA"}', axis=1) 
#         if "Class_Label_Valence" in dataset.columns: 
#             data_for_model["target"]  = dataset["Class_Label_Valence"]
#             data_for_model["continuous_target"] = dataset["Annotator_Valence"]
#         elif "Class_Label_Arousal" in dataset.columns: 
#             data_for_model["target"]  = dataset["Class_Label_Arousal"]
#             data_for_model["continuous_target"] = dataset["Annotator_Arousal"]

#         if data_for_model["features"].shape[1] >= int(inv_feature_count):
#             # STEP 2.5: RUN LOGISTIC REGRESSION TESTS
#             log_stan_results, log_inv_results, log_pca_results = run_logistic_regression_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

#             # STEP 2.6: RUN NEURAL NETWORK TESTS
#             nn_stan_results, nn_inv_results, nn_pca_results = run_neural_network_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

#             # STEP 2.7: RUN LSTM TESTS
#             lstm_stan, lstm_inv, lstm_pca = run_lstm_tests(dataset_name=DATASET_NAME, model_data=data_for_model, feature_count=inv_feature_count)

#             # STEP 2.8: SAVING RESULTS AND UPDATING CHECKPOINT FILE
#             save_results(report_file=AGAIN_REPORT_FILE, test_name=test_name, details_to_save=[log_stan_results, log_inv_results, log_pca_results, nn_stan_results, nn_inv_results, nn_pca_results, lstm_stan, lstm_inv, lstm_pca])
#             checkpoint_updater(test_name=test_name)
#         else:
#             print(f"Less features than max_invariant_features ({data_for_model["features"].shape[1]})")
#         print()

---